# Extraction / chunking / ingestion MSO

Notebook dedie aux guides MSO structures en sections plutot qu'aux FAQ natives.
Strategie retenue : conserver un pont avec les requetes utilisateurs via une pseudo-question derivee du titre de section, tout en gardant la structure documentaire reelle.

In [ ]:
import os
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

In [ ]:
%cd ..

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Iterable
from glob import glob
from urllib.parse import urlparse, urlunparse, parse_qsl, urlencode
from datetime import datetime, timezone
import hashlib
import json
import os
import re
import time
import uuid

import numpy as np
import pandas as pd
import psycopg
import requests
import torch
from psycopg.rows import dict_row
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

BASE_IN = Path(os.getenv("MSO_BASE_IN", "./data/in/MSO"))
BASE_OUT = Path(os.getenv("MSO_BASE_OUT", "./data/out/MSO"))
INPUT_FILES = [p.strip() for p in os.getenv("MSO_INPUT_FILES", "").split(",") if p.strip()]
INPUT_PATTERNS = [p.strip() for p in os.getenv("MSO_INPUT_PATTERNS", "./data/in/MSO/**/*.pdf").split(",") if p.strip()]
TXT_GLOB = [p.strip() for p in os.getenv("MSO_TXT_PATTERNS", "./data/out/MSO/**/*.txt").split(",") if p.strip()]
OUT_JSONL = os.getenv("MSO_CHUNKS_JSONL", "./data/out/chunked/mso_chunks_qna.jsonl")
OUT_DOCS_JSONL = os.getenv("MSO_DOCS_JSONL", "./data/out/chunked/mso_documents.jsonl")
OUT_SECTIONS_JSONL = os.getenv("MSO_SECTIONS_JSONL", "./data/out/chunked/mso_sections.jsonl")
OUT_JSONL_WITH_EMB = os.getenv("MSO_EMB_OUT_JSONL", "./data/out/mso_chunks_baai_bge_m3_with_emb.jsonl")
OUT_PARQUET = os.getenv("MSO_EMB_OUT_PARQUET", "./data/out/mso_chunks_baai_bge_m3.parquet")
OUT_NPY = os.getenv("MSO_EMB_OUT_NPY", "./data/out/mso_chunks_baai_bge_m3.npy")
TABLE = os.getenv("MSO_TABLE", "rag_chunks_mso")
SCHEMA = os.getenv("PGSCHEMA", "public")
MODEL_NAME = os.getenv("EMBEDDING_MODEL", "BAAI/bge-m3")
EMBED_COL = os.getenv("EMBEDDING_COLUMN", "embedding_m3")
BATCH_SIZE = int(os.getenv("MSO_EMBED_BATCH_SIZE", "64"))
NORMALIZE = True
GENERATE_BGE_SCW = os.getenv("MSO_GENERATE_BGE_SCW", "1").strip().lower() in {"1", "true", "yes", "y", "on"}
SCALEWAY_BGE_MODEL = os.getenv("SCALEWAY_BGE_MODEL", "bge-multilingual-gemma2")
SCALEWAY_BGE_BATCH_SIZE = int(os.getenv("MSO_BGE_BATCH_SIZE", "16"))
SCALEWAY_BGE_TIMEOUT = int(os.getenv("MSO_BGE_TIMEOUT", "30"))
SCALEWAY_BASE_URL = (os.getenv("SCALEWAY_BASE_URL", "https://api.scaleway.ai/11aa88cb-ec5b-4df9-bcb4-e9e82576ae58/v1") or "").rstrip("/")
SCALEWAY_API_KEY = os.getenv("SCALEWAY_API_KEY", "").strip()

SCALINGO_URL = os.getenv("SCALINGO_URL", "")
if SCALINGO_URL and not os.getenv("DATABASE_URL"):
    u = urlparse(SCALINGO_URL.replace("postgres://", "postgresql://"))
    q = dict(parse_qsl(u.query))
    q["sslmode"] = "disable"
    os.environ["DATABASE_URL"] = urlunparse(
        u._replace(
            netloc=f"{u.username}:{u.password}@127.0.0.1:10001",
            query=urlencode(q),
        )
    )

DATABASE_URL = os.getenv("DATABASE_URL", "")
DB_CFG = dict(
    host=os.getenv("PGHOST", "127.0.0.1"),
    port=int(os.getenv("PGPORT", "5432")),
    dbname=os.getenv("PGDATABASE", "postgres"),
    user=os.getenv("PGUSER", "postgres"),
    password=os.getenv("PGPASSWORD", ""),
    sslmode=os.getenv("PGSSLMODE", "require"),
)

def pg_conn():
    if DATABASE_URL:
        return psycopg.connect(DATABASE_URL, row_factory=dict_row)
    return psycopg.connect(**DB_CFG, row_factory=dict_row)

MSO_NAMESPACE = uuid.UUID("c5cdb7de-f8c4-4f8e-9b3d-c9ff7e7d4b72")

def sha1_u(s: str) -> str:
    return hashlib.sha1((s or "").encode("utf-8")).hexdigest()

def sha256_text(s: str) -> str:
    return hashlib.sha256((s or "").encode("utf-8")).hexdigest()

def stable_uuid_from_parts(namespace: uuid.UUID, *parts: object) -> str:
    key = ":".join(str(part) for part in parts)
    return str(uuid.uuid5(namespace, key))

def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def normalize_text(s: str) -> str:
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    lines = [re.sub(r"[ \t]+", " ", ln).strip() for ln in s.split("\n")]
    text = "\n".join(lines)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def expand_inputs(files: list[str], patterns: list[str]) -> list[str]:
    ordered: list[str] = []
    seen: set[str] = set()
    for f in files:
        if f and f not in seen:
            seen.add(f)
            ordered.append(f)
    for pat in patterns:
        for p in sorted(glob(pat, recursive=True)):
            if p not in seen:
                seen.add(p)
                ordered.append(p)
    return ordered

def compute_thematique(path: str) -> str:
    parts = [pp for pp in Path(path).parts if pp not in (".", "")]
    lower = [pp.lower() for pp in parts]
    if "data" in lower:
        tail = parts[lower.index("data") + 1 :]
    else:
        tail = parts[:]
    if tail and tail[0].lower() in ("in", "out"):
        tail = tail[1:]
    if tail and "." in tail[-1][1:]:
        tail = tail[:-1]
    return "/".join(tail).strip("/")

def slugify_short_id(value: str, prefix: str = "MSO") -> str:
    raw = re.sub(r"[^A-Za-z0-9]+", "_", value).strip("_") or prefix
    short_id = f"{prefix}_{raw}"[:48].rstrip("_")
    suffix = sha1_u(value)[:12]
    return f"{short_id}_{suffix}"[:64]

def estimate_tokens(text: str) -> int:
    return max(1, len(text) // 4) if text else 0

def normalize_vector(vec: list[float]) -> list[float]:
    arr = np.asarray(vec, dtype=np.float32)
    norm = float(np.linalg.norm(arr))
    return (arr / norm).tolist() if norm > 0 else arr.tolist()

def scaleway_embed_text(text: str, *, retries: int = 5) -> list[float]:
    if not SCALEWAY_API_KEY:
        raise RuntimeError("SCALEWAY_API_KEY manquante pour generer embedding_bge_scw.")
    last_error = None
    for attempt in range(retries):
        try:
            response = requests.post(
                f"{SCALEWAY_BASE_URL}/embeddings",
                headers={
                    "Authorization": f"Bearer {SCALEWAY_API_KEY}",
                    "Content-Type": "application/json",
                },
                json={"model": SCALEWAY_BGE_MODEL, "input": text},
                timeout=SCALEWAY_BGE_TIMEOUT,
            )
            if response.status_code == 429:
                time.sleep(min(30, 2 ** attempt))
                continue
            response.raise_for_status()
            payload = response.json()
            return normalize_vector(payload["data"][0]["embedding"])
        except Exception as exc:
            last_error = exc
            time.sleep(min(30, 2 ** attempt))
    if last_error is not None:
        raise last_error
    raise RuntimeError("Echec embedding_bge_scw sans erreur explicite.")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(MODEL_NAME, device=device)
print({
    "BASE_IN": str(BASE_IN),
    "BASE_OUT": str(BASE_OUT),
    "OUT_JSONL": OUT_JSONL,
    "OUT_JSONL_WITH_EMB": OUT_JSONL_WITH_EMB,
    "TABLE": TABLE,
    "MODEL_NAME": MODEL_NAME,
    "GENERATE_BGE_SCW": GENERATE_BGE_SCW,
    "SCALEWAY_BGE_MODEL": SCALEWAY_BGE_MODEL,
    "SCALEWAY_API_KEY_SET": bool(SCALEWAY_API_KEY),
    "device": device,
})

In [ ]:
def read_pdf_to_text(path: str) -> str:
    reader = PdfReader(path)
    parts: list[str] = []
    for i, page in enumerate(reader.pages, start=1):
        txt = normalize_text(page.extract_text() or "")
        if txt:
            parts.append(f"[PAGE {i}]\n{txt}")
    return normalize_text("\n\n".join(parts))

def export_pdf_texts() -> pd.DataFrame:
    inputs = [Path(p) for p in expand_inputs(INPUT_FILES, INPUT_PATTERNS)]
    rows: list[dict] = []
    for src in inputs:
        text = read_pdf_to_text(str(src))
        try:
            rel = src.relative_to(BASE_IN)
        except Exception:
            rel = Path(src.name)
        out = (BASE_OUT / rel).with_suffix(".txt")
        out.parent.mkdir(parents=True, exist_ok=True)
        out.write_text(text, encoding="utf-8")
        rows.append({
            "source_pdf": str(src),
            "output_txt": str(out),
            "chars": len(text),
            "lines": len(text.splitlines()),
        })
    return pd.DataFrame(rows)

df_exports = export_pdf_texts()
df_exports.head(20)

In [ ]:
TOC_START_PAT = re.compile(r"^table des matieres$", re.IGNORECASE)
TOC_ENTRY_PAT = re.compile(
    r"^(?:[IVXLC]+-|[A-Z]\.|\d+\.)?.{3,}\.{5,}\s*\d+\s*$",
    re.IGNORECASE,
)
HEADING_PATTERNS = [
    (1, re.compile(r"^(?P<label>[IVXLC]+-)\s*(?P<title>.+)$")),
    (2, re.compile(r"^(?P<label>[A-Z]\.)\s*(?P<title>.+)$")),
    (3, re.compile(r"^(?P<label>\d+\.)\s*(?P<title>.+)$")),
]

def strip_table_of_contents(text: str) -> str:
    lines = text.split("\n")
    out: list[str] = []
    in_toc = False
    toc_hits = 0
    for ln in lines:
        s = ln.strip()
        if TOC_START_PAT.match(s):
            in_toc = True
            toc_hits = 0
            continue
        if in_toc:
            if TOC_ENTRY_PAT.match(s) or s in {"", "[PAGE 2]", "[PAGE 3]"}:
                toc_hits += 1 if TOC_ENTRY_PAT.match(s) else 0
                continue
            if toc_hits >= 3:
                in_toc = False
            else:
                out.append(ln)
                continue
        out.append(ln)
    return normalize_text("\n".join(out))

def detect_heading(line: str):
    clean = re.sub(r"\s+", " ", line).strip()
    clean = re.sub(r"\.{4,}\s*\d+$", "", clean).strip()
    for level, pattern in HEADING_PATTERNS:
        m = pattern.match(clean)
        if m:
            title = m.group("title").strip(" -:\t")
            if title:
                return level, m.group("label"), title
    return None

def infer_user_question(section_title: str, section_path: str) -> str:
    title = re.sub(r"\s+", " ", section_title).strip(" .")
    if not title:
        return "Quelles sont les regles applicables dans cette section ?"
    if title.lower().startswith(("le ", "la ", "les ", "l'")):
        return f"Quelles sont les regles relatives a {title.lower()} ?"
    if any(token in title.lower() for token in ["procedure", "renouvellement", "recrutement", "conge", "temps partiel", "remuneration"]):
        return f"Quelle est la procedure ou les regles concernant {title.lower()} ?"
    if section_path:
        return f"Que faut-il savoir sur {title.lower()} dans le cadre de {section_path.lower()} ?"
    return f"Que faut-il savoir sur {title.lower()} ?"

def hard_wrap(text: str, max_chars: int, overlap: int) -> list[str]:
    res: list[str] = []
    i = 0
    step = max(1, max_chars - overlap)
    while i < len(text):
        res.append(text[i:i + max_chars])
        i += step
    return res

def split_on_paragraphs(text: str, max_chars: int = 1200, overlap: int = 200) -> list[str]:
    paras = [p.strip() for p in re.split(r"\n{2,}", text) if p.strip()]
    out: list[str] = []
    buf = ""
    for p in paras:
        extra = 2 if buf else 0
        if len(buf) + extra + len(p) <= max_chars:
            buf = f"{buf}\n\n{p}" if buf else p
        else:
            if buf:
                out.append(buf)
            if len(p) > max_chars:
                out.extend(hard_wrap(p, max_chars, overlap))
                buf = ""
            else:
                buf = p
    if buf:
        out.append(buf)
    return out

@dataclass
class SectionBlock:
    qa_id: str
    parent_qa_id: str | None
    parent_section_path: str | None
    section_path: str
    section_index: int
    heading_level: int
    section_title: str
    pseudo_question: str
    answer: str
    source_name: str
    thematique: str

@dataclass
class Chunk:
    hash_id: str
    qa_id: str
    parent_qa_id: str | None
    role: str
    section_path: str
    chunk_index: int
    text: str
    chunk_text: str
    source_name: str
    lang: str = "fr"
    thematique: str = ""
    source: str = "MSO"
    short_id: str = "MSO"
    references_juridiques: list[dict] | list[str] | None = None
    section_id: str | None = None
    source_document_id: str | None = None
    embedding_bge_scw: list[float] | None = None

def parse_section_blocks(text: str, source_name: str, thematique: str) -> list[SectionBlock]:
    cleaned = strip_table_of_contents(text)
    lines = cleaned.split("\n")
    blocks: list[SectionBlock] = []
    stack: list[dict] = []
    current: dict | None = None

    section_counter = 0

    def flush_current():
        nonlocal current
        if not current:
            return
        answer = normalize_text("\n".join(current["body"]))
        if answer:
            blocks.append(
                SectionBlock(
                    qa_id=current["qa_id"],
                    parent_qa_id=current["parent_qa_id"],
                    parent_section_path=current["parent_section_path"],
                    section_path=current["section_path"],
                    section_index=current["section_index"],
                    heading_level=current["level"],
                    section_title=current["title"],
                    pseudo_question=current["pseudo_question"],
                    answer=answer,
                    source_name=source_name,
                    thematique=thematique,
                )
            )
        current = None

    for raw in lines:
        line = raw.strip()
        if not line:
            if current:
                current["body"].append("")
            continue
        heading = detect_heading(line)
        if heading:
            flush_current()
            level, _, title = heading
            while stack and stack[-1]["level"] >= level:
                stack.pop()
            parent_qa_id = stack[-1]["qa_id"] if stack else None
            parent_section_path = " > ".join(x["title"] for x in stack) if stack else None
            qa_id = sha1_u(f"{source_name}|{level}|{' > '.join(x['title'] for x in stack)}|{title}")
            section_titles = [x["title"] for x in stack] + [title]
            section_path = " > ".join(section_titles)
            pseudo_question = infer_user_question(title, section_path)
            section_counter += 1
            current = {
                "qa_id": qa_id,
                "parent_qa_id": parent_qa_id,
                "parent_section_path": parent_section_path,
                "level": level,
                "title": title,
                "section_path": section_path,
                "section_index": section_counter,
                "pseudo_question": pseudo_question,
                "body": [],
            }
            stack.append({"level": level, "title": title, "qa_id": qa_id})
            continue
        if current is None:
            continue
        current["body"].append(line)

    flush_current()
    return blocks

def make_hash_id(source_name: str, qa_id: str, role: str, chunk_index: int, text: str) -> str:
    return sha1_u(f"{source_name}|{qa_id}|{role}|{chunk_index}|{text[:256]}")

def section_blocks_to_chunks(blocks: list[SectionBlock], *, doc_id: str, section_id_by_path: dict[str, str]) -> list[Chunk]:
    rows: list[Chunk] = []
    for block in blocks:
        q = block.pseudo_question
        a = block.answer
        section_id = section_id_by_path.get(block.section_path)
        prefix = f"Titre: {block.section_title}\nSection: {block.section_path}"
        q_text = f"{prefix}\nQuestion utilisateur probable: {q}"
        q_hash = make_hash_id(block.source_name, block.qa_id, "Q_ONLY", 0, q_text)
        rows.append(Chunk(q_hash, block.qa_id, block.parent_qa_id, "Q_ONLY", block.section_path, 0, q_text, q_text, block.source_name, thematique=block.thematique, section_id=section_id, source_document_id=doc_id, references_juridiques=[]))

        composite = f"{prefix}\nQuestion utilisateur probable: {q}\n\nContenu:\n{a}"[:3000]
        comp_hash = make_hash_id(block.source_name, block.qa_id, "QA_COMPOSITE", 1, composite)
        rows.append(Chunk(comp_hash, block.qa_id, block.parent_qa_id or block.qa_id, "QA_COMPOSITE", block.section_path, 1, composite, composite, block.source_name, thematique=block.thematique, section_id=section_id, source_document_id=doc_id, references_juridiques=[]))

        next_idx = 2
        for piece in split_on_paragraphs(a, max_chars=1200, overlap=200):
            atomic = f"Titre: {block.section_title}\nSection: {block.section_path}\nQuestion utilisateur probable: {q}\n\nContenu:\n{piece}"
            atomic_hash = make_hash_id(block.source_name, block.qa_id, "A_ATOMIC", next_idx, atomic)
            rows.append(Chunk(atomic_hash, block.qa_id, block.parent_qa_id or block.qa_id, "A_ATOMIC", block.section_path, next_idx, atomic, atomic, block.source_name, thematique=block.thematique, section_id=section_id, source_document_id=doc_id, references_juridiques=[]))
            next_idx += 1
    return rows

In [ ]:
def build_chunks_dataset() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    txt_files = expand_inputs([], TXT_GLOB)
    all_chunks: list[Chunk] = []
    all_documents: list[dict] = []
    all_sections: list[dict] = []
    for path_str in txt_files:
        path = Path(path_str)
        text = normalize_text(path.read_text(encoding="utf-8", errors="ignore"))
        thematique = compute_thematique(str(path))
        rel_txt = path.relative_to(BASE_OUT) if path.is_relative_to(BASE_OUT) else Path(path.name)
        pdf_path = (BASE_IN / rel_txt).with_suffix('.pdf')
        doc_title = path.stem
        short_id = slugify_short_id(str(rel_txt.with_suffix('')))
        doc_id = stable_uuid_from_parts(MSO_NAMESPACE, 'mso', short_id, str(rel_txt))
        checksum = sha256_text(text)
        blocks = parse_section_blocks(text, path.name, thematique)
        if not blocks and text:
            fallback_title = path.stem
            blocks = [
                SectionBlock(
                    qa_id=sha1_u(f"{path.name}|fallback"),
                    parent_qa_id=None,
                    parent_section_path=None,
                    section_path=fallback_title,
                    section_index=1,
                    heading_level=1,
                    section_title=fallback_title,
                    pseudo_question=infer_user_question(fallback_title, fallback_title),
                    answer=text,
                    source_name=path.name,
                    thematique=thematique,
                )
            ]
        section_id_by_path: dict[str, str] = {}
        for block in blocks:
            section_id_by_path[block.section_path] = stable_uuid_from_parts(MSO_NAMESPACE, doc_id, 'section', block.section_index)
        doc_record = {
            'doc_id': doc_id,
            'source': 'mso',
            'source_url': None,
            'storage_path': None,
            'title': doc_title,
            'full_title': doc_title,
            'short_id': short_id,
            'publisher': 'MSO',
            'doc_type': 'Guide RH',
            'last_updated_date': None,
            'publication_date': None,
            'page_count': None,
            'lang': 'fr',
            'checksum': checksum,
            'parse_version': 'extract_pdf_MSO_v1',
            'parse_model': MODEL_NAME,
            'quality_flags': {'source_format': 'pdf', 'section_aware': True},
            'doc_markdown': text,
            'doc_markdown_raw': text,
            'doc_text_hash': checksum,
            'token_count': estimate_tokens(text),
            'char_count': len(text),
            'line_count': len(text.splitlines()),
            'metadata': {
                'local_pdf_path': str(pdf_path),
                'local_txt_path': str(path),
                'thematique': thematique,
            },
            'doc_structure': {
                'section_count': len(blocks),
                'max_section_level': max((b.heading_level for b in blocks), default=0),
                'types': ['guide_mso'],
            },
            'legacy_doc_id': None,
            'created_at': utc_now_iso(),
            'updated_at': utc_now_iso(),
        }
        all_documents.append(doc_record)
        for block in blocks:
            section_markdown = f"## {block.section_title}\n\n{block.answer}".strip()
            all_sections.append({
                'section_id': section_id_by_path[block.section_path],
                'doc_id': doc_id,
                'section_index': block.section_index,
                'parent_section_id': None,
                'level': block.heading_level,
                'section_type': 'heading',
                'heading': block.section_title,
                'heading_path': block.section_path,
                'page_start': None,
                'page_end': None,
                'char_start': None,
                'char_end': None,
                'section_markdown': section_markdown,
                'token_count': estimate_tokens(section_markdown),
                'char_count': len(section_markdown),
                'teash': None,
                'doc_text_hash': checksum,
                'metadata': {'publisher': 'MSO', 'doc_short_id': short_id},
                'created_at': doc_record['created_at'],
                'updated_at': doc_record['updated_at'],
                'text_hash': sha256_text(section_markdown),
                'is_indexable': True,
                'references_juridiques': [],
            })
        chunks = section_blocks_to_chunks(blocks, doc_id=doc_id, section_id_by_path=section_id_by_path)
        for chunk in chunks:
            chunk.short_id = short_id
        all_chunks.extend(chunks)
        print(f"{path.name}: {len(blocks)} sections -> {len(chunks)} chunks")

    out_path = Path(OUT_JSONL)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("w", encoding="utf-8") as f:
        for chunk in all_chunks:
            f.write(json.dumps(asdict(chunk), ensure_ascii=False) + "\n")
    docs_path = Path(OUT_DOCS_JSONL)
    docs_path.parent.mkdir(parents=True, exist_ok=True)
    with docs_path.open('w', encoding='utf-8') as f:
        for rec in all_documents:
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')
    sections_path = Path(OUT_SECTIONS_JSONL)
    sections_path.parent.mkdir(parents=True, exist_ok=True)
    with sections_path.open('w', encoding='utf-8') as f:
        for rec in all_sections:
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')
    print(f"JSONL saved to {out_path} ({len(all_chunks)} rows)")
    return pd.DataFrame([asdict(c) for c in all_chunks]), pd.DataFrame(all_documents), pd.DataFrame(all_sections)

df_chunks, df_documents, df_sections = build_chunks_dataset()
with pd.option_context("display.max_colwidth", 140):
    display(df_chunks[["source_name", "role", "section_path", "chunk_index", "text"]].head(20))

In [ ]:
def format_passage(text: str) -> str:
    if MODEL_NAME.startswith("intfloat/multilingual-e5"):
        return f"passage: {text or ''}"
    return text or ""

def add_embeddings() -> pd.DataFrame:
    rows = [json.loads(line) for line in Path(OUT_JSONL).read_text(encoding="utf-8").splitlines() if line.strip()]
    df = pd.DataFrame(rows)
    assert not df.empty, "Aucun chunk a embedder."
    corpus = [format_passage(t) for t in df["chunk_text"].astype(str).tolist()]
    vecs_all = []
    for i in range(0, len(corpus), BATCH_SIZE):
        batch = corpus[i:i + BATCH_SIZE]
        vecs = model.encode(
            batch,
            batch_size=len(batch),
            convert_to_numpy=True,
            normalize_embeddings=NORMALIZE,
            show_progress_bar=True,
        )
        vecs_all.append(vecs)
    emb = np.vstack(vecs_all).astype(np.float32)
    df[EMBED_COL] = [v.tolist() for v in emb]
    if GENERATE_BGE_SCW:
        bge_vectors: list[list[float]] = []
        for i in range(0, len(corpus), SCALEWAY_BGE_BATCH_SIZE):
            batch = corpus[i:i + SCALEWAY_BGE_BATCH_SIZE]
            print({"embedding_bge_scw_batch_start": i, "batch_size": len(batch)})
            for text in batch:
                bge_vectors.append(scaleway_embed_text(text))
        df["embedding_bge_scw"] = bge_vectors
    elif "embedding_bge_scw" not in df.columns:
        df["embedding_bge_scw"] = None
    if "references_juridiques" not in df.columns:
        df["references_juridiques"] = None
    if "section_id" not in df.columns:
        df["section_id"] = None
    if "source_document_id" not in df.columns:
        df["source_document_id"] = None
    Path(OUT_JSONL_WITH_EMB).parent.mkdir(parents=True, exist_ok=True)
    with open(OUT_JSONL_WITH_EMB, "w", encoding="utf-8") as f:
        for rec in df.to_dict(orient="records"):
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    np.save(OUT_NPY, emb)
    parquet_df = df.copy()
    parquet_df[EMBED_COL] = parquet_df[EMBED_COL].apply(json.dumps)
    if "embedding_bge_scw" in parquet_df.columns:
        parquet_df["embedding_bge_scw"] = parquet_df["embedding_bge_scw"].apply(lambda v: json.dumps(v) if isinstance(v, list) else None)
    parquet_df.to_parquet(OUT_PARQUET, index=False)
    print({"jsonl_with_emb": OUT_JSONL_WITH_EMB, "npy": OUT_NPY, "parquet": OUT_PARQUET, "shape": emb.shape, "embedding_bge_scw_generated": bool(GENERATE_BGE_SCW)})
    return df

df_emb = add_embeddings()
df_emb[["source_name", "role", "section_path", "chunk_index"]].head(20)

In [ ]:
CREATE_DOCUMENTS_SQL = """
CREATE TABLE IF NOT EXISTS \"public\".\"rag_documents\" (
    doc_id UUID PRIMARY KEY,
    source TEXT,
    source_url TEXT,
    storage_path TEXT,
    title TEXT,
    full_title TEXT,
    short_id VARCHAR(64),
    publisher TEXT,
    doc_type TEXT,
    last_updated_date DATE,
    publication_date DATE,
    page_count INTEGER,
    lang TEXT,
    checksum TEXT,
    parse_version TEXT,
    parse_model TEXT,
    quality_flags JSONB,
    doc_markdown TEXT,
    doc_text_hash TEXT,
    metadata JSONB,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    doc_markdown_raw TEXT,
    token_count INTEGER,
    char_count INTEGER,
    line_count INTEGER,
    doc_structure JSONB,
    legacy_doc_id UUID
);
CREATE INDEX IF NOT EXISTS idx_rag_documents_short_id ON \"public\".\"rag_documents\" (short_id);
"""

CREATE_SECTIONS_SQL = """
CREATE TABLE IF NOT EXISTS \"public\".\"rag_sections\" (
    section_id UUID PRIMARY KEY,
    doc_id UUID,
    section_index INTEGER,
    parent_section_id UUID,
    level SMALLINT,
    section_type TEXT,
    heading TEXT,
    heading_path TEXT,
    page_start INTEGER,
    page_end INTEGER,
    char_start INTEGER,
    char_end INTEGER,
    section_markdown TEXT,
    token_count INTEGER,
    char_count INTEGER,
    teash TEXT,
    doc_text_hash TEXT,
    metadata JSONB,
    created_at TIMESTAMPTZ,
    updated_at TIMESTAMPTZ,
    text_hash TEXT,
    is_indexable BOOLEAN,
    references_juridiques JSONB
);
CREATE INDEX IF NOT EXISTS idx_rag_sections_doc_id ON \"public\".\"rag_sections\" (doc_id);
CREATE INDEX IF NOT EXISTS idx_rag_sections_refs ON \"public\".\"rag_sections\" USING GIN (references_juridiques);
"""

CREATE_LEGACY_DOCUMENTS_SQL = """
CREATE TABLE IF NOT EXISTS \"public\".\"documents\" (
    id UUID PRIMARY KEY,
    filename TEXT,
    mime_type TEXT,
    size_bytes BIGINT,
    sha256_hex TEXT,
    source_name TEXT,
    created_at TIMESTAMPTZ,
    data BYTEA
);
"""

CREATE_TABLE_SQL = f"""
CREATE EXTENSION IF NOT EXISTS vector;
CREATE TABLE IF NOT EXISTS \"{SCHEMA}\".\"{TABLE}\" (
    hash_id VARCHAR(64) PRIMARY KEY,
    qa_id TEXT,
    parent_qa_id TEXT,
    source_name VARCHAR(255),
    section_path TEXT,
    role TEXT,
    chunk_index INTEGER,
    text TEXT,
    chunk_text TEXT,
    lang TEXT DEFAULT 'fr',
    thematique TEXT,
    source TEXT,
    short_id VARCHAR(64),
    references_juridiques JSONB,
    section_id UUID,
    source_document_id UUID,
    created_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP,
    updated_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP,
    text_tsv tsvector GENERATED ALWAYS AS (
        to_tsvector('french', coalesce(section_path, '') || ' ' || coalesce(chunk_text, ''))
    ) STORED,
    embedding_m3 vector(1024),
    embedding_bge_scw vector(3584)
);
CREATE INDEX IF NOT EXISTS idx_{TABLE}_short_id ON \"{SCHEMA}\".\"{TABLE}\" (short_id);
CREATE INDEX IF NOT EXISTS idx_{TABLE}_section_id ON \"{SCHEMA}\".\"{TABLE}\" (section_id);
CREATE INDEX IF NOT EXISTS idx_{TABLE}_tsv ON \"{SCHEMA}\".\"{TABLE}\" USING GIN (text_tsv);
CREATE INDEX IF NOT EXISTS idx_{TABLE}_{EMBED_COL} ON \"{SCHEMA}\".\"{TABLE}\" USING ivfflat ({EMBED_COL} vector_cosine_ops) WITH (lists = 100);
CREATE INDEX IF NOT EXISTS idx_{TABLE}_embedding_bge_scw ON \"{SCHEMA}\".\"{TABLE}\" USING ivfflat (embedding_bge_scw vector_cosine_ops) WITH (lists = 100);
"""

UPSERT_LEGACY_DOCUMENTS_SQL = """
INSERT INTO \"public\".\"documents\"
(id, filename, mime_type, size_bytes, sha256_hex, source_name, created_at, data)
VALUES
(%(id)s::uuid, %(filename)s, %(mime_type)s, %(size_bytes)s, %(sha256_hex)s, %(source_name)s, %(created_at)s::timestamptz, %(data)s)
ON CONFLICT (id) DO UPDATE SET
    filename = EXCLUDED.filename,
    mime_type = EXCLUDED.mime_type,
    size_bytes = EXCLUDED.size_bytes,
    sha256_hex = EXCLUDED.sha256_hex,
    source_name = EXCLUDED.source_name,
    data = EXCLUDED.data;
"""

UPSERT_DOCUMENTS_SQL = """
INSERT INTO \"public\".\"rag_documents\"
(doc_id, source, source_url, storage_path, title, full_title, publisher, doc_type, last_updated_date, publication_date, page_count, lang, checksum, parse_version, parse_model, quality_flags, doc_markdown, doc_text_hash, metadata, created_at, updated_at, doc_markdown_raw, token_count, char_count, line_count, short_id, doc_structure, legacy_doc_id)
VALUES
(%(doc_id)s::uuid, %(source)s, %(source_url)s, %(storage_path)s, %(title)s, %(full_title)s, %(publisher)s, %(doc_type)s, %(last_updated_date)s, %(publication_date)s, %(page_count)s, %(lang)s, %(checksum)s, %(parse_version)s, %(parse_model)s, %(quality_flags)s::jsonb, %(doc_markdown)s, %(doc_text_hash)s, %(metadata)s::jsonb, %(created_at)s::timestamp, %(updated_at)s::timestamp, %(doc_markdown_raw)s, %(token_count)s, %(char_count)s, %(line_count)s, %(short_id)s, %(doc_structure)s::jsonb, %(legacy_doc_id)s::uuid)
ON CONFLICT (doc_id) DO UPDATE SET
    source = EXCLUDED.source,
    source_url = EXCLUDED.source_url,
    storage_path = EXCLUDED.storage_path,
    title = EXCLUDED.title,
    full_title = EXCLUDED.full_title,
    publisher = EXCLUDED.publisher,
    doc_type = EXCLUDED.doc_type,
    last_updated_date = EXCLUDED.last_updated_date,
    publication_date = EXCLUDED.publication_date,
    page_count = EXCLUDED.page_count,
    lang = EXCLUDED.lang,
    checksum = EXCLUDED.checksum,
    parse_version = EXCLUDED.parse_version,
    parse_model = EXCLUDED.parse_model,
    quality_flags = EXCLUDED.quality_flags,
    doc_markdown = EXCLUDED.doc_markdown,
    doc_text_hash = EXCLUDED.doc_text_hash,
    metadata = EXCLUDED.metadata,
    updated_at = EXCLUDED.updated_at,
    doc_markdown_raw = EXCLUDED.doc_markdown_raw,
    token_count = EXCLUDED.token_count,
    char_count = EXCLUDED.char_count,
    line_count = EXCLUDED.line_count,
    short_id = EXCLUDED.short_id,
    doc_structure = EXCLUDED.doc_structure,
    legacy_doc_id = EXCLUDED.legacy_doc_id;
"""

UPSERT_SECTIONS_SQL = """
INSERT INTO \"public\".\"rag_sections\"
(section_id, doc_id, section_index, parent_section_id, level, section_type, heading, heading_path, page_start, page_end, char_start, char_end, section_markdown, token_count, char_count, teash, doc_text_hash, metadata, created_at, updated_at, text_hash, is_indexable, references_juridiques)
VALUES
(%(section_id)s::uuid, %(doc_id)s::uuid, %(section_index)s, %(parent_section_id)s::uuid, %(level)s, %(section_type)s, %(heading)s, %(heading_path)s, %(page_start)s, %(page_end)s, %(char_start)s, %(char_end)s, %(section_markdown)s, %(token_count)s, %(char_count)s, %(teash)s, %(doc_text_hash)s, %(metadata)s::jsonb, %(created_at)s::timestamptz, %(updated_at)s::timestamptz, %(text_hash)s, %(is_indexable)s, %(references_juridiques)s::jsonb)
ON CONFLICT (section_id) DO UPDATE SET
    doc_id = EXCLUDED.doc_id,
    section_index = EXCLUDED.section_index,
    parent_section_id = EXCLUDED.parent_section_id,
    section_type = EXCLUDED.section_type,
    level = EXCLUDED.level,
    heading = EXCLUDED.heading,
    heading_path = EXCLUDED.heading_path,
    page_start = EXCLUDED.page_start,
    page_end = EXCLUDED.page_end,
    char_start = EXCLUDED.char_start,
    char_end = EXCLUDED.char_end,
    section_markdown = EXCLUDED.section_markdown,
    token_count = EXCLUDED.token_count,
    char_count = EXCLUDED.char_count,
    teash = EXCLUDED.teash,
    doc_text_hash = EXCLUDED.doc_text_hash,
    metadata = EXCLUDED.metadata,
    created_at = EXCLUDED.created_at,
    updated_at = EXCLUDED.updated_at,
    text_hash = EXCLUDED.text_hash,
    is_indexable = EXCLUDED.is_indexable,
    references_juridiques = EXCLUDED.references_juridiques;
"""

UPSERT_SQL = f"""
INSERT INTO \"{SCHEMA}\".\"{TABLE}\"
(hash_id, qa_id, parent_qa_id, source_name, section_path, role, chunk_index, text, chunk_text, lang, thematique, short_id, source, references_juridiques, section_id, source_document_id, embedding_m3, embedding_bge_scw)
VALUES
(%(hash_id)s, %(qa_id)s, %(parent_qa_id)s, %(source_name)s, %(section_path)s, %(role)s, %(chunk_index)s, %(text)s, %(chunk_text)s, %(lang)s, %(thematique)s, %(short_id)s, %(source)s, %(references_juridiques)s::jsonb, %(section_id)s::uuid, %(source_document_id)s::uuid, %(embedding_m3)s::vector, %(embedding_bge_scw)s::vector)
ON CONFLICT (hash_id) DO UPDATE SET
    qa_id = EXCLUDED.qa_id,
    parent_qa_id = EXCLUDED.parent_qa_id,
    source_name = EXCLUDED.source_name,
    section_path = EXCLUDED.section_path,
    role = EXCLUDED.role,
    chunk_index = EXCLUDED.chunk_index,
    text = EXCLUDED.text,
    chunk_text = EXCLUDED.chunk_text,
    lang = EXCLUDED.lang,
    thematique = EXCLUDED.thematique,
    short_id = EXCLUDED.short_id,
    source = EXCLUDED.source,
    references_juridiques = EXCLUDED.references_juridiques,
    section_id = EXCLUDED.section_id,
    source_document_id = EXCLUDED.source_document_id,
    embedding_m3 = EXCLUDED.embedding_m3,
    embedding_bge_scw = EXCLUDED.embedding_bge_scw,
    updated_at = CURRENT_TIMESTAMP;
"""

def vec_to_pgvector(v: Iterable[float]) -> str:
    vals = list(v)
    return "[" + ",".join(f"{float(x):.7f}" for x in vals) + "]"

def make_legacy_document_payload(doc_record: dict) -> dict:
    meta = doc_record.get('metadata') or {}
    pdf_path = Path(meta.get('local_pdf_path') or meta.get('original_local_pdf_path') or '')
    if not pdf_path.exists():
        raise FileNotFoundError(f'PDF source introuvable: {pdf_path}')
    pdf_bytes = pdf_path.read_bytes()
    pdf_sha = hashlib.sha256(pdf_bytes).hexdigest()
    legacy_doc_id = stable_uuid_from_parts(MSO_NAMESPACE, 'legacy_pdf', pdf_path.name, pdf_sha)
    return {
        'id': legacy_doc_id,
        'filename': pdf_path.name,
        'mime_type': 'application/pdf',
        'size_bytes': len(pdf_bytes),
        'sha256_hex': pdf_sha,
        'source_name': 'MSO',
        'created_at': utc_now_iso(),
        'data': pdf_bytes,
    }

def upsert_to_db(batch_size: int = 1000) -> None:
    documents = [json.loads(line) for line in Path(OUT_DOCS_JSONL).read_text(encoding='utf-8').splitlines() if line.strip()]
    sections = [json.loads(line) for line in Path(OUT_SECTIONS_JSONL).read_text(encoding='utf-8').splitlines() if line.strip()]
    rows = [json.loads(line) for line in Path(OUT_JSONL_WITH_EMB).read_text(encoding="utf-8").splitlines() if line.strip()]
    df = pd.DataFrame(rows)
    assert not df.empty, "Aucune ligne a inserer."
    df["embedding_m3"] = df[EMBED_COL].apply(vec_to_pgvector)
    df["embedding_bge_scw"] = df["embedding_bge_scw"].apply(lambda v: vec_to_pgvector(v) if isinstance(v, list) else None)
    legacy_documents = []
    for rec in documents:
        legacy_payload = make_legacy_document_payload(rec)
        legacy_documents.append(legacy_payload)
        rec['legacy_doc_id'] = legacy_payload['id']
        rec['source_url'] = None
        rec['metadata'] = {
            **(rec.get('metadata') or {}),
            'original_local_pdf_path': (rec.get('metadata') or {}).get('local_pdf_path'),
        }
        rec['quality_flags'] = json.dumps(rec.get('quality_flags')) if rec.get('quality_flags') is not None else None
        rec['metadata'] = json.dumps(rec.get('metadata')) if rec.get('metadata') is not None else None
        rec['doc_structure'] = json.dumps(rec.get('doc_structure')) if rec.get('doc_structure') is not None else None
    for rec in sections:
        rec['parent_section_id'] = None
        rec['section_type'] = 'heading'
        rec['metadata'] = json.dumps(rec.get('metadata') or {})
        rec['references_juridiques'] = json.dumps(rec.get('references_juridiques') or [])
    if 'references_juridiques' in df.columns:
        df['references_juridiques'] = df['references_juridiques'].apply(lambda v: json.dumps(v or []))
    if 'source_document_id' in df.columns and documents:
        doc_id = documents[0]['doc_id']
        df['source_document_id'] = df['source_document_id'].fillna(doc_id)
    with pg_conn() as con, con.cursor() as cur:
        for stmt in [s.strip() for s in CREATE_LEGACY_DOCUMENTS_SQL.split(";") if s.strip()]:
            cur.execute(stmt)
        for stmt in [s.strip() for s in CREATE_DOCUMENTS_SQL.split(";") if s.strip()]:
            cur.execute(stmt)
        for stmt in [s.strip() for s in CREATE_SECTIONS_SQL.split(";") if s.strip()]:
            cur.execute(stmt)
        for stmt in [s.strip() for s in CREATE_TABLE_SQL.split(";") if s.strip()]:
            cur.execute(stmt)
        con.commit()
        if legacy_documents:
            cur.executemany(UPSERT_LEGACY_DOCUMENTS_SQL, legacy_documents)
            con.commit()
        if documents:
            cur.executemany(UPSERT_DOCUMENTS_SQL, documents)
            con.commit()
        if sections:
            cur.executemany(UPSERT_SECTIONS_SQL, sections)
            con.commit()
        for i in range(0, len(df), batch_size):
            batch = df.iloc[i:i + batch_size].to_dict(orient="records")
            cur.executemany(UPSERT_SQL, batch)
            con.commit()
    print(f"Upsert termine: docs={len(documents)} sections={len(sections)} chunks={len(df)} dans {SCHEMA}.{TABLE}")

In [ ]:
# Decommenter pour pousser en base.
# upsert_to_db()